# Task 6 — Agents that reproduce the main Duffing results

```text
Task 5 → orchestrator ↔ scientist
                      ↔ general
                      ↔ executor → compact notebook → Python runner
                                                        ↓
                                              scores, table, and plot
```

The reference notebook is now used as a guide to the experiment, not as a template to copy. The generated notebook only needs to reproduce the main workflow and evidence: prepare the data, train one baseline acceleration network, perform the uninterrupted free run, compare its RMSE with the zero baseline, and plot the test prediction.

The executor creates and runs a new notebook on every attempt. A result counts only when the notebook executes successfully and prints real train and test RMSE values. The graph still has at most eight orchestrator decisions and each executor run has a five-minute limit.


In [ ]:
import os
import sys
import json
import math
import getpass
import operator
import tempfile
import subprocess
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
import io
import re
import time
import signal
import tokenize
import traceback
import shutil
import nbformat


In [ ]:
class State(TypedDict):
    dialogue: Annotated[list[dict], operator.add]
    next_role: str
    decisions: int
    has_result: bool
    latest_draft: str


## Model answers

`Decision` is the orchestrator's routing answer. `Reply` contains a specialist response. `NotebookDraft` contains the complete ordered notebook for the first generation. `CellRepair` contains one corrected code cell for a later repair. Execution outputs always come from the local runner.


In [ ]:
class Decision(BaseModel):
    next_role: Literal["scientist", "general", "executor", "finish"]
    instruction: str
    reason: str

class Reply(BaseModel):
    text: str

class NotebookCell(BaseModel):
    cell_type: Literal["markdown", "code"]
    source: str

class NotebookDraft(BaseModel):
    cells: list[NotebookCell]

class CellRepair(BaseModel):
    cell_number: int
    corrected_source: str
    reason: str


## Model

Use standard Gemini 3.8 Flash through the Gemini Developer API

In [ ]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI API key: ")

model = ChatGoogleGenerativeAI(model="gemini-3.8-flash", vertexai=False, timeout=180, max_retries=2)

decision_model = model.with_structured_output(Decision, method="json_schema")
reply_model = model.with_structured_output(Reply, method="json_schema")
code_model = model.with_structured_output(NotebookDraft, method="json_schema")
repair_model = model.with_structured_output(CellRepair, method="json_schema")

print("Configured model:", model.model)


## Dialogue


In [ ]:
def dialogue_text(state):
    return json.dumps(state["dialogue"], ensure_ascii=False)


## Working folder

Each run gets a new folder. Generated scripts, their printed output, errors, plots, and the final dialogue go there.


## Concrete experiment contract

This plain-text contract gives the scientist and executor the exact first-run choices. They organize and implement the notebook, but they do not have to guess the dataset protocol, model architecture, scaling, optimizer, rollout length, time budget, evaluation, or required files.


In [ ]:
experiment_contract = """
Build one compact notebook for this exact baseline experiment.

1. Load and prepare the data
- Set DATA_PATH to the supplied MAT-file path.
- Load V1 as input u and V2 as output y with scipy.io.loadmat.
- Squeeze both arrays, convert them to NumPy float32, and verify that they are finite and contain at least 46746 samples.
- Take zero-based slice [42650:46746], producing exactly 4096 samples.
- Subtract the mean of the complete 4096-sample u excerpt from u and independently subtract the mean of the complete y excerpt from y.
- Use fs=610.35 Hz and dt=1/fs.
- Use samples 0:3072 as training and 3072:4096 as test.

2. Compute training-only scales
- y_scale=max(std(y_train), 1e-6).
- u_scale=max(std(u_train), 1e-6).
- Estimate training velocity with diff(y_train)/dt and set v_scale=max(std(estimated_velocity), 1e-6).
- Estimate training acceleration with diff(estimated_velocity)/dt and set a_scale=max(std(estimated_acceleration), 1e-6).
- Keep RK4 states in physical units. Apply these scales only inside the neural network.

3. Build the single baseline model
- Use a PyTorch AccelerationMLP with layers 3 -> 64 -> 64 -> 1 and Tanh after both hidden layers.
- The three inputs are y, v, and u. Normalize them by y_scale, v_scale, and u_scale.
- Multiply the final scalar network output by a_scale so the returned value is acceleration in physical units.
- Use torch.float32 and seed NumPy and PyTorch with 42.

4. Implement differentiable RK4
- Represent the state with y and v, where dy/dt=v and dv/dt=AccelerationMLP(y,v,u).
- Define rk4_step(model, y, v, u_current, u_next, dt) and return y_next, v_next.
- Use u_current for the first RK4 stage, (u_current+u_next)/2 for both midpoint stages, and u_next for the last stage.
- Do not detach tensors during training.

5. Train the baseline
- Use Adam with learning rate 1e-3 and gradient clipping at norm 1.0.
- Use training rollout length 32 and mean squared voltage error as the loss.
- For every update, choose a random section wholly inside the 3072 training samples.
- Initialize that section with its measured training y and forward-difference training velocity. After initialization, predict the complete section through RK4 without feeding later measured y values.
- Stop after 800 updates or 175 seconds, whichever happens first. Check elapsed time between updates.
- Print progress occasionally and stop with a clear error if the loss becomes non-finite.

6. Perform the required continuous evaluation
- Initialize once with y[0] and v0=(y[1]-y[0])/dt.
- Run one uninterrupted rollout over all 4096 input samples.
- The rollout may use known u but must never use measured test y and must not reset at sample 3072.
- Reject the result if any predicted value is non-finite.
- Compute train RMSE on predictions 0:3072, test RMSE on predictions 3072:4096, and zero-prediction test RMSE as sqrt(mean(y_test**2)).

7. Save and print the evidence
- Save comparison.csv with exactly these columns: model, hidden_layers, learning_rate, rollout_length, updates, training_seconds, train_rmse, test_rmse, zero_test_rmse, status.
- Save test_prediction.png showing measured and predicted y for all 1024 test samples with labeled axes and a legend.
- Print the baseline settings and all three RMSE values.
- Finish by printing TRAIN_RMSE=<finite number> and TEST_RMSE=<finite number> from the real continuous rollout.
- If training or evaluation fails, raise a clear error and do not print the two success markers.
"""


In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="task6_", dir=Path.cwd()))
print(workdir)


## Coding tool: notebook → Python runner → executed notebook

The tool checks that generated Python cells compile and that the output is a valid notebook. It no longer requires exact headings, a fixed cell count, or a long Markdown word count.

The runner executes the notebook in a fresh kernel using the same Python interpreter. It saves completed outputs and tracebacks. The outer five-minute timeout covers the complete run.

`check_result` accepts a run only when execution succeeds and the notebook prints finite, nonnegative `TRAIN_RMSE` and `TEST_RMSE` values. This proves that code ran; the resulting plot and comparison table still need scientific inspection.


In [ ]:
RUNNER_SOURCE = r'''
import os
import sys
from pathlib import Path
import nbformat
from nbclient import NotebookClient
from jupyter_client import KernelManager

notebook_path = Path(sys.argv[1]).resolve()
notebook = nbformat.read(notebook_path, as_version=4)

class SamePythonKernel(KernelManager):
    def format_kernel_cmd(self, extra_arguments=None):
        return [sys.executable, "-m", "ipykernel_launcher", "-f", self.connection_file]

def save_notebook():
    temporary = notebook_path.with_suffix(".saving.ipynb")
    nbformat.write(notebook, temporary)
    os.replace(temporary, notebook_path)

def report_cell(cell, cell_index, **kwargs):
    save_notebook()
    print(f"Completed cell {cell_index + 1}", flush=True)
    for output in cell.get("outputs", []):
        if output.output_type == "stream":
            target = sys.stderr if output.name == "stderr" else sys.stdout
            print(output.text, end="", file=target, flush=True)

client = NotebookClient(
    notebook,
    timeout=270,
    startup_timeout=30,
    allow_errors=False,
    kernel_manager_class=SamePythonKernel,
    resources={"metadata": {"path": str(notebook_path.parent)}},
    on_cell_executed=report_cell,
)

try:
    client.execute()
finally:
    save_notebook()
'''


def prepare_notebook(draft):
    if not draft.cells:
        raise ValueError("The generated notebook has no cells.")

    cells = []
    code_cell_count = 0
    for cell_number, item in enumerate(draft.cells, start=1):
        if item.cell_type == "markdown":
            cells.append(nbformat.v4.new_markdown_cell(item.source))
        else:
            compile(item.source, f"generated notebook cell {cell_number}", "exec")
            cells.append(nbformat.v4.new_code_cell(item.source))
            code_cell_count += 1

    if code_cell_count == 0:
        raise ValueError("The generated notebook must contain executable Python.")

    notebook = nbformat.v4.new_notebook(cells=cells)
    notebook.metadata.kernelspec = {
        "name": "python3", "display_name": "Python 3", "language": "python"
    }
    nbformat.validate(notebook)
    return notebook


def coding_tool(draft, number):
    attempt = workdir / f"attempt_{number}"
    attempt.mkdir(exist_ok=False)
    draft_path = attempt / "draft.json"
    draft_path.write_text(draft.model_dump_json(indent=2), encoding="utf-8")
    notebook_path = attempt / "Duffing_generated.ipynb"
    script_path = attempt / "run_notebook.py"

    try:
        notebook = prepare_notebook(draft)
        nbformat.write(notebook, notebook_path)
        script_path.write_text(RUNNER_SOURCE.strip() + "\n", encoding="utf-8")
    except Exception:
        return {
            "exit_code": None, "stdout": "", "stderr": traceback.format_exc(),
            "notebook": None, "attempt": str(attempt), "stage": "validation"
        }

    output_path = attempt / "stdout.txt"
    error_path = attempt / "stderr.txt"
    process_options = {}
    if os.name == "nt":
        process_options["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    else:
        process_options["start_new_session"] = True
    environment = os.environ.copy()
    environment["TASK6_RUN_MODE"] = "quick"
    started = time.monotonic()
    timed_out = False

    with output_path.open("w", encoding="utf-8") as out, error_path.open("w", encoding="utf-8") as err:
        process = subprocess.Popen(
            [sys.executable, "-X", "utf8", "-u", str(script_path), str(notebook_path)],
            cwd=attempt, stdout=out, stderr=err, env=environment, **process_options
        )
        try:
            exit_code = process.wait(timeout=300)
        except subprocess.TimeoutExpired:
            timed_out = True
            if os.name == "nt":
                subprocess.run(
                    ["taskkill", "/PID", str(process.pid), "/T", "/F"],
                    capture_output=True, check=False, timeout=15
                )
            else:
                os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=15)
            exit_code = None

    stderr = error_path.read_text(encoding="utf-8", errors="replace")
    if timed_out:
        stderr += "\nThe five-minute limit stopped the runner and its kernel. Earlier completed cell outputs were saved."
    result = {
        "exit_code": exit_code,
        "stdout": output_path.read_text(encoding="utf-8", errors="replace"),
        "stderr": stderr,
        "notebook": str(notebook_path),
        "attempt": str(attempt),
        "seconds": round(time.monotonic() - started, 2),
        "stage": "execution"
    }
    (attempt / "execution.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def check_result(run):
    if run["exit_code"] != 0 or not run.get("notebook"):
        return None
    notebook = nbformat.read(run["notebook"], as_version=4)
    code_cells = [cell for cell in notebook.cells if cell.cell_type == "code"]
    if any(cell.execution_count is None for cell in code_cells):
        return None
    if any(output.output_type == "error" for cell in code_cells for output in cell.outputs):
        return None
    scores = {}
    for line in run["stdout"].splitlines():
        for label in ("TRAIN_RMSE", "TEST_RMSE"):
            if line.startswith(label + "="):
                try:
                    scores[label] = float(line.split("=", 1)[1])
                except ValueError:
                    return None
    if set(scores) != {"TRAIN_RMSE", "TEST_RMSE"}:
        return None
    if not all(math.isfinite(value) and value >= 0 for value in scores.values()):
        return None
    return scores


## Nodes

### Orchestrator

The model reads the dialogue and chooses the next role. Its decision is added to the dialogue, so the chosen specialist sees its instruction. Python rejects `finish` if no execution result exists; the orchestrator must choose again. Rejected decisions still count toward eight.


In [ ]:
MAX_DECISIONS = 8

def orchestrator(state: State):
    print("---Orchestrator---")

    if state["decisions"] >= MAX_DECISIONS:
        text = json.dumps({
            "next_role": "finish",
            "instruction": "Stop because all eight orchestrator decisions have been used.",
            "reason": "The decision limit has been reached."
        })
        print(text)
        return {
            "dialogue": [{"role": "orchestrator", "text": text}],
            "next_role": "finish"
        }

    answer = decision_model.invoke([
        ("system", """Choose the next role, its instruction, and a one-sentence reason.
Use scientist once to create the scientific plan and step interfaces.
Use executor to implement that plan or repair a failed generated cell.
Use general only when a missing background explanation prevents progress.
You provide routing instructions only; leave scientific planning to scientist and code to executor.
If a complete scientist plan already exists, do not request it again.
Choose finish only after executor reports exit code zero and verified train and test RMSE.
Never invent results. You have at most eight decisions."""),
        ("human", dialogue_text(state) + f"\nDecision {state['decisions'] + 1} of {MAX_DECISIONS}.")
    ])
    next_role = answer.next_role
    text = answer.model_dump_json()

    if next_role == "finish" and not state["has_result"]:
        has_plan = any(message["role"] == "scientist" for message in state["dialogue"])
        next_role = "executor" if has_plan else "scientist"
        text += f"\nFinish rejected because no verified result exists. Routing to {next_role}."

    print(text)
    return {
        "dialogue": [{"role": "orchestrator", "text": text}],
        "next_role": next_role,
        "decisions": state["decisions"] + 1
    }


### Scientist

The scientist produces the scientific strategy and a compact dependency contract. It chooses the implementation steps and defines what each step reads, creates, and proves. The executor decides how to map those steps into Markdown and code cells.


In [ ]:
def scientist(state: State):
    print("---Scientist---")
    answer = reply_model.invoke([
        ("system", """You are scientist. Produce the scientific plan and interface contract.
Leave all Python source and implementation to executor. Treat the experiment contract in
the user task as the single source of truth.

Begin with a short strategy. Then describe the smallest useful sequence of implementation
steps. You choose the steps; do not prescribe notebook headings or Markdown cells.

For every step use this format:
STEP <number>: <short purpose>
READS: <values created earlier, including relevant type and dimensions>
CREATES: <new values, including relevant type and dimensions>
INTERFACE: <function signatures or output-file columns>
CHECK: <printed value, assertion, table, or file that proves the step worked>

Choose names once and use them consistently. A step may read only values created earlier.
The essential external dimensions are: complete signals have 4096 samples, training signals
have 3072, test signals have 1024, the acceleration model receives [y,v,u] and returns one
acceleration value, and the final predicted output contains 4096 samples. Choose compatible
internal batch shapes and give exact signatures for the model call, RK4 step, training
function, and continuous evaluation function.

Use state (y,v), where v=dy/dt. Plan alpha_theta as one small acceleration MLP receiving
y, v, and u. The state equations are dy/dt=v and dv/dt=alpha_theta(y,v,u), and RK4 advances
that state. Plan one baseline configuration for this first working run. Give it a hard
training budget of at most 175 seconds, checked between optimizer updates, so the complete
300-second run retains time for evaluation, plotting, and saving.

Keep the plan limited to data preparation, the baseline model, training, uninterrupted
continuous evaluation, RMSE reporting, comparison.csv, and test_prediction.png.
Return the complete plan in this one response. Never present an unexecuted value as measured."""),
        ("human", dialogue_text(state))
    ])
    return {"dialogue": [{"role": "scientist", "text": answer.text}]}


### General agent

Same shape as scientist, but different instructions. This role explains concepts rather than creating the fitting plan.


In [ ]:
def general(state: State):
    print("---General---")
    answer = reply_model.invoke([
        ("system", """Follow the orchestrator's instruction as general agent.
Explain background concepts, dataset records, or error metrics simply.
Do not write the fitting plan or code. You have no coding tool.
Plans belong to scientist and traceback repairs belong to executor.
Never invent measured results."""),
        ("human", dialogue_text(state))
    ])
    return {"dialogue": [{"role": "general", "text": answer.text}]}


### Executor

For the first attempt, the executor turns the scientist's steps into a complete notebook. A notebook may contain both Markdown and code cells, while each individual cell is one type. After a failure, the executor receives the complete saved draft and returns one corrected code cell; the tool preserves every other cell.


In [ ]:
def executor(state: State):
    print("---Executor: building or repairing the results notebook---")
    if not any(message["role"] == "scientist" for message in state["dialogue"]):
        return {"dialogue": [{"role": "executor", "text": "Ask scientist for one complete plan first."}]}

    latest_draft = state.get("latest_draft", "")
    dialogue = dialogue_text(state)

    try:
        if latest_draft:
            previous_draft = NotebookDraft.model_validate_json(latest_draft)
            repair = repair_model.invoke([
                ("system", """You are executor repairing one failed Python cell in a saved notebook.
Return a CellRepair containing the one-based cell number, the complete corrected source for
that cell, and a short reason. Read the traceback and identify the first error originating
from generated notebook code. Preserve the cell's existing public variable names, function
signatures, outputs, and the scientist's interfaces so later cells remain compatible.

Use four spaces for every indentation level and never use tab characters. Check every function,
loop, condition, and context manager for consistent indentation. The corrected cell must compile
as ordinary Python. Fix the root cause without changing unrelated behavior, hiding exceptions,
inventing values, shortening the scored interval, or printing false success markers.
If a rollout is non-finite, correct the relevant numerical or training logic; do not replace
non-finite values with fabricated predictions. Return only the requested CellRepair."""),
                ("human", dialogue + "\n\nComplete saved NotebookDraft:\n" + latest_draft)
            ])
            if not 1 <= repair.cell_number <= len(previous_draft.cells):
                raise ValueError(f"Repair cell {repair.cell_number} is outside the notebook.")
            target_cell = previous_draft.cells[repair.cell_number - 1]
            if target_cell.cell_type != "code":
                raise ValueError(f"Repair cell {repair.cell_number} is not a code cell.")
            compile(repair.corrected_source, f"repaired notebook cell {repair.cell_number}", "exec")
            draft = previous_draft.model_copy(deep=True)
            draft.cells[repair.cell_number - 1].source = repair.corrected_source
            repair_summary = repair.model_dump()
        else:
            draft = code_model.invoke([
                ("system", """You are executor. Return a complete NotebookDraft whose ordered cells
implement the scientist's plan and the experiment contract in the user task. The notebook may
contain both Markdown and code cells; set each cell_type to either 'markdown' or 'code'.

Follow every READS, CREATES, INTERFACE, and CHECK entry from the scientist. Order cells so every
input exists before it is read. Use brief Markdown only where it helps identify a major step.
Return complete ordinary Python that runs offline from top to bottom in a fresh kernel. Use only
the supplied local data and installed packages. Do not require interactive input, package
installation, network access, child processes, or placeholder code.

Use four spaces for each indentation level and never use tab characters. Check the indentation
of every function, loop, condition, and context manager. Each code cell must compile independently.

Implement one baseline configuration with a hard training budget of at most 175 seconds, checked
between optimizer updates. Preserve time for the uninterrupted 4096-sample rollout, scoring,
comparison.csv, and test_prediction.png. Never reset at the train/test boundary or feed measured
test y into the rollout. Reject non-finite predictions and do not suppress unexpected exceptions.
Print TRAIN_RMSE=<number> and TEST_RMSE=<number> only after real successful evaluation."""),
                ("human", dialogue)
            ])
            repair_summary = None

        latest_draft = draft.model_dump_json()
        run = coding_tool(draft, state["decisions"])
        scores = check_result(run)
        manifest = []
        for cell_number, cell in enumerate(draft.cells, start=1):
            first_line = next(
                (line.strip() for line in cell.source.splitlines() if line.strip()),
                "empty cell"
            )
            manifest.append({
                "cell": cell_number,
                "type": cell.cell_type,
                "starts_with": first_line[:120]
            })
        execution_summary = {
            "exit_code": run.get("exit_code"),
            "stage": run.get("stage"),
            "attempt": run.get("attempt"),
            "notebook": run.get("notebook"),
            "seconds": run.get("seconds"),
            "stdout_tail": run.get("stdout", "")[-2000:],
            "stderr_tail": run.get("stderr", "")[-6000:]
        }
        report = {
            "execution": execution_summary,
            "verified_printed_scores": scores,
            "repair": repair_summary,
            "cell_manifest": manifest
        }
        text = json.dumps(report, ensure_ascii=False)
        print(json.dumps(report, indent=2, ensure_ascii=False))
    except Exception:
        scores = None
        text = "Notebook generation or repair failed:\n" + traceback.format_exc()
        print(text)

    return {
        "dialogue": [{"role": "executor", "text": text}],
        "has_result": state["has_result"] or scores is not None,
        "latest_draft": latest_draft
    }


## Edges

Take state as input and return the next node name.

The model already chose a role. This function reads that choice and enforces the stop condition. At the eighth decision it stops immediately, without running another specialist. That is a limit stop, not necessarily successful completion.


In [ ]:
def choose_next(state: State):
    if state["next_role"] == "finish":
        return END
    return state["next_role"]


## Building the Graph

Register the four functions and connect their turns. Specialists return to orchestrator. The self-edge lets orchestrator retry an invalid early finish. `compile()` prepares the graph; it does not run it.


In [ ]:
builder = StateGraph(State)

builder.add_node("orchestrator", orchestrator)
builder.add_node("scientist", scientist)
builder.add_node("general", general)
builder.add_node("executor", executor)

builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", choose_next, {
    "scientist": "scientist",
    "general": "general",
    "executor": "executor",
    "orchestrator": "orchestrator",
    END: END
})
builder.add_edge("scientist", "orchestrator")
builder.add_edge("general", "orchestrator")
builder.add_edge("executor", "orchestrator")

graph = builder.compile()


## Visualize the Graph

The same display call as your earlier notebook. This optional cell uses an online diagram renderer; skip it if that service is unavailable.


In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))


## Task 5

The user talks to the orchestrator by supplying the first message. This text gives the actual task, not a ready-made solution. The data path below points to the file in your `Uni_HW` folder; change it if you run elsewhere.

The original assignment explicitly selects settings by test RMSE. We preserve that requirement here.


In [ ]:
import importlib.util

required_modules = [
    "numpy", "pandas", "scipy", "matplotlib", "torch",
    "nbformat", "nbclient", "jupyter_client", "ipykernel"
]
missing_modules = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
if missing_modules:
    missing_text = ", ".join(missing_modules)
    raise ModuleNotFoundError(
        f"The current kernel is missing: {missing_text}. "
        f"Install them into this Python environment before calling Gemini: {sys.executable}"
    )

data_path = Path(r"C:\Users\saroy\Codes\Uni_HW\SilverboxFiles\SNLS80mV.mat")
if not data_path.is_file():
    raise FileNotFoundError(data_path)

task = f"""Create and execute a compact notebook for Task 5.

The data file is:
{data_path}

Follow this concrete experiment protocol:
{experiment_contract}

The first run intentionally uses one baseline model to verify the complete generated-notebook
pipeline. Keep explanations short and report only values produced by execution.
"""


## Graph Inference

This starts model calls and actual Python execution. The initial dialogue contains only the user's task. Nodes add messages as they run.

The graph's step limit of 30 is separate from our eight orchestrator decisions. The complete result is saved for the assignment report. A decision-limit stop may be incomplete even if an earlier attempt produced a number.


In [ ]:
result = graph.invoke({
    "dialogue": [{"role": "user", "text": task}],
    "next_role": "orchestrator",
    "decisions": 0,
    "has_result": False,
    "latest_draft": ""
}, config={"recursion_limit": 30})

(workdir / "dialogue.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

if result["next_role"] == "finish" and result["has_result"]:
    print("Finished with an execution result.")
else:
    print("Stopped at the decision limit.")
print("Files:", workdir)


## Read the Dialogue and open the generated notebook

Each executor message lists its attempt directory and notebook path. Open `Duffing_generated.ipynb` from a successful attempt to read explanations alongside real tables, plots, and execution outputs. Failed attempts remain available for debugging.

The attempt directory also contains `draft.json`, `run_notebook.py`, stdout/stderr logs, and `execution.json`. The generated notebook's own results folder contains its model checkpoints, plots, tables, and ZIP.

For submission, inspect the scientific code, show the actual agent process and free-run score, and include your GitHub link. A five-minute quick run demonstrates the system; report its actual training budget and limitations.

No Gemini calls or Duffing training have been run during this edit. The output notebook is created when you run Graph Inference.


In [ ]:
for message in result["dialogue"]:
    print("\n" + message["role"].upper())
    print(message["text"])


# Part 2 — An alternative system that generates Python files

## Why this version uses a Python file

Part 1 asks the executor to organize, create, and run a Jupyter notebook. This alternative keeps the same four-agent workflow and scientific protocol but asks the executor for one ordinary `solution.py` file. It is useful when we want to inspect the scientific implementation separately from notebook assembly and presentation.

The earlier configuration combined strict structural validation with a large output specification. This section changes the output format and reduces the first experiment to one concrete baseline. It does not treat the earlier outcome as a model-capability failure.

Part 2 is self-contained and can run from a fresh kernel. Run its cells in order and use only its Graph Inference cell for this option.

## The concept

```text
Task 5 → Orchestrator ↔ Scientist: implementation and interface plan
                     ↔ General: background when needed
                     ↔ Executor: create or repair solution.py → run it
                                                               ↓
                                              scores, CSV, plot, or traceback
                     → END after verified results or the decision limit
```

Only executor generates and runs Python. The full latest source is kept in graph state for repairs; the shared dialogue contains compact execution evidence.


## 1. Imports

These imports define the LangGraph roles, structured model responses, file execution, timeouts, and result validation. Part 2 runs an ordinary Python file and therefore does not need notebook-generation libraries.


In [ ]:
import os
import sys
import csv
import json
import math
import time
import signal
import getpass
import operator
import tempfile
import subprocess
import traceback
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END


## 2. State

`dialogue` stores compact role messages, `next_role` stores the orchestrator's destination, `decisions` counts orchestrator choices, and `has_result` records verified execution. `latest_program` privately preserves the complete generated source so executor can repair it without copying it through every role message.


In [ ]:
class State(TypedDict):
    dialogue: Annotated[list[dict], operator.add]
    next_role: str
    decisions: int
    has_result: bool
    latest_program: str


## 3. Model answer formats

`Decision` routes the graph and `Reply` contains a specialist response. `PythonProgram` contains the complete source for the first attempt. `PythonRepair` contains a complete corrected source and a short explanation for later attempts.


In [ ]:
class Decision(BaseModel):
    next_role: Literal["scientist", "general", "executor", "finish"]
    instruction: str
    reason: str

class Reply(BaseModel):
    text: str

class PythonProgram(BaseModel):
    code: str

class PythonRepair(BaseModel):
    code: str
    reason: str


## 4. Gemini 3.8 Flash

All roles use the same model with different instructions and answer formats. The API key comes from `GOOGLE_API_KEY`; if it is missing, the hidden prompt asks for it. There is no model-ID question.

This cell prepares the clients. The model calls start when Graph Inference runs. Choosing the model does not change the API project's billing tier or available quota.


In [ ]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI API key: ")

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash", vertexai=False, timeout=180, max_retries=2
)
decision_model = model.with_structured_output(Decision, method="json_schema")
reply_model = model.with_structured_output(Reply, method="json_schema")
python_code_model = model.with_structured_output(PythonProgram, method="json_schema")
python_repair_model = model.with_structured_output(PythonRepair, method="json_schema")

print("Configured model:", model.model)


## 5. Dialogue

The model needs to read what the previous roles said. This helper converts our list of messages into text while preserving the role labels.


In [ ]:
def dialogue_text(state):
    return json.dumps(state["dialogue"], ensure_ascii=False)


## 6. Coding tool — write and run `solution.py`

The tool saves the exact model-generated source as both `original_source.txt` and `solution.py`, then compiles it before execution. Keeping the executed source unchanged makes traceback line numbers match the source used for repair.

The script runs with the current kernel's Python interpreter. Standard output, errors, timing, stage, and exit code are saved for every attempt in `execution.json`. The complete process has a five-minute limit. A timeout stops the generated process tree while retaining the files and logs already produced.


In [ ]:
def python_coding_tool(source, number):
    attempt = simple_workdir / f"attempt_{number}"
    attempt.mkdir(exist_ok=False)
    script = attempt / "solution.py"
    original = attempt / "original_source.txt"
    original.write_text(source, encoding="utf-8")
    script.write_text(source, encoding="utf-8")
    started = time.monotonic()

    try:
        compile(source, str(script), "exec")
    except Exception:
        run = {
            "exit_code": None,
            "stdout": "",
            "stderr": traceback.format_exc(),
            "script": str(script),
            "seconds": round(time.monotonic() - started, 2),
            "stage": "syntax_check"
        }
        (attempt / "execution.json").write_text(
            json.dumps(run, indent=2), encoding="utf-8"
        )
        return run

    output_path = attempt / "stdout.txt"
    error_path = attempt / "stderr.txt"
    process_options = {}
    if os.name == "nt":
        process_options["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    else:
        process_options["start_new_session"] = True
    timed_out = False
    print("Running:", script)

    with output_path.open("w", encoding="utf-8") as out, error_path.open("w", encoding="utf-8") as err:
        process = subprocess.Popen(
            [sys.executable, "-X", "utf8", "-u", str(script)],
            cwd=attempt, stdout=out, stderr=err, **process_options
        )
        try:
            exit_code = process.wait(timeout=300)
        except subprocess.TimeoutExpired:
            timed_out = True
            if os.name == "nt":
                subprocess.run(
                    ["taskkill", "/PID", str(process.pid), "/T", "/F"],
                    capture_output=True, check=False, timeout=15
                )
            else:
                os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=15)
            exit_code = None

    stderr = error_path.read_text(encoding="utf-8", errors="replace")
    if timed_out:
        stderr += "\nStopped: the script exceeded five minutes."
    run = {
        "exit_code": exit_code,
        "stdout": output_path.read_text(encoding="utf-8", errors="replace"),
        "stderr": stderr,
        "script": str(script),
        "seconds": round(time.monotonic() - started, 2),
        "stage": "execution"
    }
    (attempt / "execution.json").write_text(
        json.dumps(run, indent=2), encoding="utf-8"
    )
    return run


## 7. Validate scores and artifacts

A successful attempt must exit normally, print finite nonnegative `TRAIN_RMSE` and `TEST_RMSE` values, save a nonempty `test_prediction.png`, and save `comparison.csv` with the exact contract columns and at least one result row. This checks execution evidence and required deliverables; the source and plot still need scientific review.


In [ ]:
def python_scores(run):
    if run["exit_code"] != 0:
        return None

    scores = {}
    for line in run["stdout"].splitlines():
        for name in ("TRAIN_RMSE", "TEST_RMSE"):
            if line.startswith(name + "="):
                try:
                    scores[name] = float(line.split("=", 1)[1])
                except ValueError:
                    return None
    if set(scores) != {"TRAIN_RMSE", "TEST_RMSE"}:
        return None
    if not all(math.isfinite(value) and value >= 0 for value in scores.values()):
        return None

    attempt = Path(run["script"]).parent
    comparison_path = attempt / "comparison.csv"
    plot_path = attempt / "test_prediction.png"
    if not comparison_path.is_file() or not plot_path.is_file():
        return None
    if comparison_path.stat().st_size == 0 or plot_path.stat().st_size == 0:
        return None

    required_columns = [
        "model", "hidden_layers", "learning_rate", "rollout_length",
        "updates", "training_seconds", "train_rmse", "test_rmse",
        "zero_test_rmse", "status"
    ]
    with comparison_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
    if reader.fieldnames != required_columns or not rows:
        return None

    return {
        **scores,
        "comparison": str(comparison_path),
        "plot": str(plot_path)
    }


## 8. Nodes

Each node reads shared state and returns an update. The orchestrator routes work, scientist defines a compact component contract, general supplies background only when it blocks progress, and executor creates or repairs the script.

### Orchestrator

The orchestrator requests one scientific plan and then delegates implementation or repair. An early `finish` is redirected to the missing role, and all eight allowed decisions can dispatch work.


In [ ]:
PYTHON_MAX_DECISIONS = 8

def python_orchestrator(state: State):
    print("---Orchestrator: Python version---")

    if state["decisions"] >= PYTHON_MAX_DECISIONS:
        text = json.dumps({
            "next_role": "finish",
            "instruction": "Stop because all eight orchestrator decisions have been used.",
            "reason": "The decision limit has been reached."
        })
        print(text)
        return {
            "dialogue": [{"role": "orchestrator", "text": text}],
            "next_role": "finish"
        }

    answer = decision_model.invoke([
        ("system", """Coordinate the Task 5 Python-file workflow. Choose the next role,
its instruction, and a one-sentence reason. Use scientist once for the scientific and
function-interface plan. Use executor for first implementation and later repairs. Use
general only when a missing background explanation prevents progress. You route work only;
leave scientific planning to scientist and Python to executor. Do not request the same plan
twice. Choose finish only after executor reports exit code zero, verified scores, the required
CSV, and the required plot. Never invent results. You have at most eight decisions."""),
        ("human", dialogue_text(state) + f"\nDecision {state['decisions'] + 1} of {PYTHON_MAX_DECISIONS}.")
    ])
    destination = answer.next_role
    text = answer.model_dump_json()
    if destination == "finish" and not state["has_result"]:
        has_plan = any(message["role"] == "scientist" for message in state["dialogue"])
        destination = "executor" if has_plan else "scientist"
        text += f"\nFinish rejected because no verified result exists. Routing to {destination}."
    print(text)
    return {
        "dialogue": [{"role": "orchestrator", "text": text}],
        "next_role": destination,
        "decisions": state["decisions"] + 1
    }


### Scientist

Scientist produces one implementation strategy and a function-interface contract for the concrete baseline. It specifies dependencies and checks without writing Python or deciding presentation details.


In [ ]:
def python_scientist(state: State):
    print("---Scientist: Python plan---")
    answer = reply_model.invoke([
        ("system", """Produce the scientific plan and function-interface contract for the
concrete experiment protocol in the user task. Leave all Python source to executor.

Begin with a short strategy. Then describe the smallest useful sequence of components.
For every component use this format:
COMPONENT: <short purpose>
READS: <existing values with relevant type and dimensions>
CREATES: <new values with relevant type and dimensions>
INTERFACE: <exact function signature or output-file columns>
CHECK: <printed value, assertion, score, or file proving it worked>

Choose names once and use them consistently. Define exact signatures for data loading,
the acceleration model call, RK4 step, short-rollout training, and continuous evaluation.
Use the task's one baseline architecture and hyperparameters without adding alternatives.
Keep the plan limited to data preparation, scaling, model, training, uninterrupted rollout,
RMSE calculation, comparison.csv, and test_prediction.png. Return the complete plan in this
one response. Never write Python or present unexecuted values as measurements."""),
        ("human", dialogue_text(state))
    ])
    return {"dialogue": [{"role": "scientist", "text": answer.text}]}


### General agent

General provides a short background explanation only when it is necessary for progress. It does not plan the experiment, implement Python, or claim measured results.


In [ ]:
def general(state: State):
    print("---General---")
    answer = reply_model.invoke([
        ("system", """Answer the orchestrator's specific background question simply.
Limit your response to concepts needed for the current task. Leave the scientific plan
to scientist and all implementation and debugging to executor. Never invent results."""),
        ("human", dialogue_text(state))
    ])
    return {"dialogue": [{"role": "general", "text": answer.text}]}


### Executor

On the first turn, executor generates one complete script from the scientist's interfaces and the concrete task protocol. On later turns, it receives the previous complete source privately, reads the actual traceback, and returns a corrected script. Shared dialogue receives only execution summaries and artifact paths.


In [ ]:
def python_executor(state: State):
    print("---Executor: writing or repairing a Python file---")
    if not any(message["role"] == "scientist" for message in state["dialogue"]):
        return {"dialogue": [{"role": "executor", "text": "Ask scientist for one complete plan first."}]}

    latest_program = state.get("latest_program", "")
    dialogue = dialogue_text(state)

    try:
        if latest_program:
            repair = python_repair_model.invoke([
                ("system", """Repair the saved Python program using the actual execution stage,
traceback, and stdout. Return a PythonRepair with the complete corrected source and a short
reason. Preserve the scientist's function interfaces, the concrete experiment protocol,
and every working part. Use four spaces per indentation level and never use tabs. Fix the
first root cause in generated code. Return complete executable Python, not a patch or code
fence. Do not hide exceptions, fabricate predictions, shorten the scored interval, print
false success markers, add experiments, or install packages."""),
                ("human", dialogue + "\n\nComplete saved Python source:\n" + latest_program)
            ])
            source = repair.code
            repair_summary = repair.reason
        else:
            program = python_code_model.invoke([
                ("system", """Return one complete ordinary Python script in the code field.
Follow the concrete experiment protocol in the user task and every interface in the scientist's
plan. Use simple functions and one small nn.Module. Return executable Python only, not a notebook,
Markdown, code fence, interactive prompt, installer, network call, child process, or placeholder.

Use four spaces per indentation level and never use tabs. Ensure every function, loop, condition,
and context manager is consistently indented. Use the supplied local MAT file and write outputs
only in the current working directory. Keep the program deterministic with the required seeds.

Implement the one baseline configuration exactly. Check the 175-second training budget between
updates and preserve time for the uninterrupted 4096-sample evaluation and artifact saving.
Do not reset at the train/test boundary or feed measured test output into the rollout. Reject
non-finite training or predictions without suppressing unexpected exceptions. Save the exact
comparison.csv schema and test_prediction.png required by the task. Print TRAIN_RMSE=<number>
and TEST_RMSE=<number> only after a real successful continuous evaluation."""),
                ("human", dialogue)
            ])
            source = program.code
            repair_summary = None

        latest_program = source
        run = python_coding_tool(source, state["decisions"])
        scores = python_scores(run)
        execution_summary = {
            "exit_code": run.get("exit_code"),
            "stage": run.get("stage"),
            "script": run.get("script"),
            "seconds": run.get("seconds"),
            "stdout_tail": run.get("stdout", "")[-2000:],
            "stderr_tail": run.get("stderr", "")[-6000:]
        }
        report = {
            "execution": execution_summary,
            "verified_result": scores,
            "repair_reason": repair_summary
        }
        text = json.dumps(report, ensure_ascii=False)
        print(json.dumps(report, indent=2, ensure_ascii=False))
    except Exception:
        scores = None
        text = "Python generation or repair failed:\n" + traceback.format_exc()
        print(text)

    return {
        "dialogue": [{"role": "executor", "text": text}],
        "has_result": state["has_result"] or scores is not None,
        "latest_program": latest_program
    }


## 9. Edges

The conditional edge follows the orchestrator's selected role or ends on `finish`. The decision-limit check occurs inside the orchestrator, so the eighth selected specialist is allowed to run before the graph stops.


In [ ]:
def choose_next(state: State):
    if state["next_role"] == "finish":
        return END
    return state["next_role"]


## 10. Building the graph

START leads to orchestrator, the selected specialist returns to orchestrator, and `compile()` prepares the runnable graph. Defining this graph does not call Gemini or execute generated Python.


In [ ]:
python_builder = StateGraph(State)
python_builder.add_node("orchestrator", python_orchestrator)
python_builder.add_node("scientist", python_scientist)
python_builder.add_node("general", general)
python_builder.add_node("executor", python_executor)
python_builder.add_edge(START, "orchestrator")
python_builder.add_conditional_edges("orchestrator", choose_next, {
    "scientist": "scientist",
    "general": "general",
    "executor": "executor",
    "orchestrator": "orchestrator",
    END: END
})
python_builder.add_edge("scientist", "orchestrator")
python_builder.add_edge("general", "orchestrator")
python_builder.add_edge("executor", "orchestrator")
python_graph = python_builder.compile()


## 11. Concrete Python-file task

Part 2 repeats the concrete one-model baseline so it can run independently from a fresh kernel. The dependency preflight runs before any Gemini request. The first successful run proves the Python-file pipeline; architecture comparison can be added afterwards.


In [ ]:
import importlib.util

required_modules = [
    "numpy", "pandas", "scipy", "matplotlib", "torch"
]
missing_modules = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
if missing_modules:
    missing_text = ", ".join(missing_modules)
    raise ModuleNotFoundError(
        f"The current kernel is missing: {missing_text}. "
        f"Install them into this Python environment before calling Gemini: {sys.executable}"
    )

data_path = Path(r"C:\Users\saroy\Codes\Uni_HW\SilverboxFiles\SNLS80mV.mat")
if not data_path.is_file():
    raise FileNotFoundError(data_path)

python_experiment_contract = """
Build one ordinary Python file for this exact baseline experiment.

1. Load V1 as u and V2 as y from the supplied MAT file with scipy.io.loadmat. Squeeze,
convert to NumPy float32, verify finite values, take slice [42650:46746], and independently
center u and y over those 4096 samples. Use fs=610.35, dt=1/fs, 3072 training samples,
and 1024 test samples.

2. From training data only, use y_scale=max(std(y_train),1e-6),
u_scale=max(std(u_train),1e-6), v_scale=max(std(diff(y_train)/dt),1e-6), and
a_scale=max(std(diff(diff(y_train)/dt)/dt),1e-6). Keep RK4 in physical units and apply
scaling only inside the network.

3. Use torch.float32 with NumPy and PyTorch seed 42. Build AccelerationMLP with
3 -> 64 -> 64 -> 1 layers and Tanh after both hidden layers. Normalize [y,v,u] by the
three input scales and multiply the scalar output by a_scale.

4. Define differentiable rk4_step(model,y,v,u_current,u_next,dt). Use u_current in the
first stage, the average of current and next u in both midpoint stages, and u_next in the
last stage. Return y_next and v_next without detaching during training.

5. Train with Adam learning rate 1e-3, gradient clipping norm 1.0, rollout length 32,
and mean squared voltage error. Each update starts from a random section wholly inside
training, using its measured first y and forward-difference velocity only for initialization.
After that initialization, predict the full section without later measured y. Stop after
800 updates or 175 seconds, checking time between updates. Reject non-finite loss.

6. Evaluate once from y[0] and (y[1]-y[0])/dt over all 4096 inputs. Use known u, never
feed measured test y, and never reset at index 3072. Reject non-finite predictions. Compute
train RMSE, test RMSE, and sqrt(mean(y_test**2)) as the zero-prediction test baseline.

7. Save comparison.csv with exactly these columns in this order: model, hidden_layers,
learning_rate, rollout_length, updates, training_seconds, train_rmse, test_rmse,
zero_test_rmse, status. Save a nonempty test_prediction.png for all 1024 test samples.
Print the settings and all scores, then print TRAIN_RMSE=<finite number> and
TEST_RMSE=<finite number>. On failure, raise an error and print no success markers.
"""

python_task = f"""Create and execute solution.py for Task 5.

The data file is:
{data_path.resolve()}

Follow this concrete experiment protocol:
{python_experiment_contract}

Return an executable Python file only. Keep explanations in the agent dialogue brief.
"""


## 12. Graph Inference — start the Python-file run

This cell creates a new `task6_python_...` folder and starts the agent workflow. Each attempt saves the exact source, execution metadata, stdout, and stderr. A successful attempt also contains the verified comparison CSV and prediction plot.

Outputs displayed from an older run describe that older configuration. Running this cell creates a new independent run using the current prompts and contract.


In [ ]:
simple_workdir = Path(tempfile.mkdtemp(prefix="task6_python_", dir=Path.cwd()))
print("Python-run files:", simple_workdir)
python_result = python_graph.invoke({
    "dialogue": [{"role": "user", "text": python_task}],
    "next_role": "orchestrator",
    "decisions": 0,
    "has_result": False,
    "latest_program": ""
}, config={"recursion_limit": 30})

(simple_workdir / "dialogue.json").write_text(
    json.dumps(python_result, indent=2), encoding="utf-8"
)
if python_result["next_role"] == "finish" and python_result["has_result"]:
    print("Finished with verified Python execution results.")
else:
    print("Stopped without a verified result; inspect the saved attempts.")
print("Python-run files:", simple_workdir)


## 13. Read the agent dialogue

This final view shows routing decisions, the scientist's interface plan, compact executor reports, repair reasons, and verified artifact paths. The complete latest Python source remains available in `python_result["latest_program"]` and in each attempt folder.


In [ ]:
for message in python_result["dialogue"]:
    print("\n" + message["role"].upper())
    print(message["text"])
